# #11993 tl.argmax/argmin NaN — main 빌드 검증 + 후보 수정 실험

**사람이 할 일: 런타임을 GPU(L4)로 바꾸고 `Run all`만 누르고, 끝날 때까지 탭을 열어 둔다.**
`drive.mount`/`input`은 쓰지 않는다. 전 셀이 무인 실행이다.

순서: 환경 → 의존성+clone → main 소스 빌드(20~60분 이상 걸릴 수 있음) → baseline(main) 측정 →
후보 수정 적용 → 재측정 → 기존 reduce 테스트. 빌드 셀은 재실행해도 이미 빌드됐으면 건너뛴다.


In [ ]:
import subprocess, sys, os
def sh(cmd, timeout=600):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return (r.stdout + r.stderr).strip()
print(sh("nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader"))
print("nproc", os.cpu_count())
print(sh("free -g | head -2"))
print(sh("python --version; gcc --version | head -1; df -h /content | tail -1"))


In [ ]:
import os, subprocess
def sh(cmd, timeout=1800):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return (r.stdout + r.stderr).strip()
os.chdir("/content")   # cwd must not be inside the repo we are about to delete
print(sh('pip install -q "cmake>=3.20,<4.0" ninja "nanobind==2.10.2" setuptools wheel pytest scipy numpy lit 2>&1 | tail -3'))
if not os.path.isdir("/content/triton/.git"):
    print(sh("git clone --depth 1 https://github.com/triton-lang/triton.git /content/triton 2>&1 | tail -2"))
print("HEAD", sh("cd /content/triton && git rev-parse HEAD && git log -1 --format=%cI"))
print(sh("find /content/triton -maxdepth 1 -iname 'setup.py' -o -maxdepth 1 -iname 'pyproject.toml'"))


In [ ]:
import os, sys, subprocess, time
def built():
    r = subprocess.run([sys.executable, "-c", "import triton;print(triton.__version__, triton.__file__)"],
                       cwd="/tmp", capture_output=True, text=True)
    return (r.returncode == 0 and "/content/triton" in r.stdout), (r.stdout + r.stderr).strip()
ok, msg = built()
if ok:
    print("already built:", msg)
else:
    env = dict(os.environ); env["MAX_JOBS"] = str(os.cpu_count())
    p = subprocess.Popen("cd /content/triton && pip install -e . --no-build-isolation -v > /content/build.log 2>&1",
                         shell=True, env=env)
    t0 = time.time()
    while p.poll() is None:
        time.sleep(180)
        tail = subprocess.run("tail -n 1 /content/build.log | cut -c1-220", shell=True, capture_output=True, text=True).stdout.strip()
        print(f"[{int(time.time()-t0)//60} min] {tail}", flush=True)
    print("build rc", p.returncode, f"elapsed {int(time.time()-t0)//60} min")
    if p.returncode != 0:
        print(subprocess.run("tail -n 60 /content/build.log", shell=True, capture_output=True, text=True).stdout)
        raise SystemExit("build failed; see tail above")
ok, msg = built()
print("import check:", ok, msg)
assert ok, "triton not importable from /content/triton"


In [ ]:
import os
os.makedirs("/content/w", exist_ok=True)
WORKER = r"""
import os, sys, json
import torch, triton, triton.language as tl

interp = os.environ.get("TRITON_INTERPRET") == "1"
dev = "cpu" if interp else "cuda"

@triton.jit
def kf(x_ptr, i_ptr, v_ptr, N: tl.constexpr, TB: tl.constexpr):
    x = tl.load(x_ptr + tl.arange(0, N))
    tl.store(i_ptr + 0, tl.argmax(x, 0, tie_break_left=TB))
    tl.store(i_ptr + 1, tl.argmin(x, 0, tie_break_left=TB))
    tl.store(v_ptr + 0, tl.max(x, 0))
    tl.store(v_ptr + 1, tl.min(x, 0))

def launch(x, N, TB, nw):
    i = torch.zeros(2, dtype=torch.int32, device=dev)
    v = torch.zeros(2, dtype=x.dtype, device=dev)
    h = kf[(1,)](x.to(dev), i, v, N=N, TB=TB, num_warps=nw)
    if not interp:
        torch.cuda.synchronize()
    return i.cpu().tolist(), v.cpu().tolist(), h

g = torch.Generator().manual_seed(1)
out = {"triton": triton.__version__, "interp": interp, "cfg": {}, "nonan": {}, "allnan": {}, "ptx": {}}
nws = (1,) if interp else (1, 4)
for N in (8, 32, 128, 1024):
    for nw in nws:
        for TB in (True, False):
            mx = dict(ok=0, nan_idx=0, wrong_finite=0); mn = dict(ok=0, nan_idx=0, wrong_finite=0)
            trials = 4 if N <= 128 else 2
            for _ in range(trials):
                base = torch.randperm(N, generator=g).float() - N / 2
                pos = list(range(N)) if N <= 128 else sorted(set([0, 1, N // 2, N - 2, N - 1] + torch.randint(0, N, (35,), generator=g).tolist()))
                for p in pos:
                    x = base.clone(); x[p] = float("nan")
                    xf = base.clone(); xf[p] = -1e30; emax = int(xf.argmax())
                    xf = base.clone(); xf[p] = 1e30; emin = int(xf.argmin())
                    i, v, _h = launch(x, N, TB, nw)
                    for d, got, exp in ((mx, i[0], emax), (mn, i[1], emin)):
                        if got == exp: d["ok"] += 1
                        elif got == p: d["nan_idx"] += 1
                        else: d["wrong_finite"] += 1
            out["cfg"]["N%d_nw%d_TB%s" % (N, nw, TB)] = {"argmax": mx, "argmin": mn}

# regression: inputs without NaN, with ties
for N in (8, 32, 128):
    for TB in (True, False):
        bad_idx = 0; bad_val = 0; tot = 0
        for _ in range(64):
            x = torch.randint(0, 4, (N,), generator=g).float()
            i, v, _h = launch(x, N, TB, 4 if not interp else 1)
            tot += 1
            if TB and (i[0] != int(x.argmax()) or i[1] != int(x.argmin())): bad_idx += 1
            if float(x[i[0]]) != float(x.max()) or float(x[i[1]]) != float(x.min()): bad_val += 1
        out["nonan"]["N%d_TB%s" % (N, TB)] = dict(total=tot, bad_leftmost=bad_idx, bad_value=bad_val)

# behaviour on all-NaN input (must not change with the fix)
for N in (8, 32, 128):
    for TB in (True, False):
        x = torch.full((N,), float("nan"))
        i, v, _h = launch(x, N, TB, 4 if not interp else 1)
        out["allnan"]["N%d_TB%s" % (N, TB)] = i

# codegen size proxy (compiled only)
if not interp:
    for name, x in (("fp32", torch.randn(128)), ("int32", torch.randint(0, 100, (128,), dtype=torch.int32))):
        _i, _v, h = launch(x, 128, True, 4)
        ptx = h.asm["ptx"]
        out["ptx"][name] = dict(lines=len(ptx.splitlines()), setp=ptx.count("setp."), selp=ptx.count("selp."))
print("JSON:" + json.dumps(out))
"""
open("/content/w/worker.py", "w").write(WORKER)
import ast; ast.parse(WORKER)
print("worker written", len(WORKER))


In [ ]:
import os, sys, json, subprocess
RESULTS = {}
def run_stage(label, timeout=2400):
    res = {}
    for mode, extra in (("interp", {"TRITON_INTERPRET": "1"}), ("gpu", {})):
        env = dict(os.environ); env.update(extra)
        r = subprocess.run([sys.executable, "/content/w/worker.py"], cwd="/tmp", env=env,
                           capture_output=True, text=True, timeout=timeout)
        line = [l for l in r.stdout.splitlines() if l.startswith("JSON:")]
        if r.returncode != 0 or not line:
            print("FAILED", label, mode, "rc", r.returncode, "\nSTDOUT:", r.stdout[-1500:], "\nSTDERR:", r.stderr[-2500:])
            res[mode] = {"error": r.stderr[-1500:]}
            continue
        res[mode] = json.loads(line[0][5:])
    RESULTS[label] = res
    json.dump(RESULTS, open("/content/results_11993.json", "w"))
    return res

def summarize(label):
    for mode, r in RESULTS[label].items():
        if "error" in r:
            print(f"== {label}/{mode}: ERROR"); continue
        print(f"== {label}/{mode} triton={r['triton']}")
        for which in ("argmax", "argmin"):
            agg = dict(ok=0, nan_idx=0, wrong_finite=0)
            for c in r["cfg"].values():
                for k in agg: agg[k] += c[which][k]
            tot = sum(agg.values())
            print(f"  {which}: total={tot} ok={agg['ok']} nan_idx={agg['nan_idx']} wrong_finite={agg['wrong_finite']}")
        print("  nonan:", {k: (v['bad_leftmost'], v['bad_value']) for k, v in r["nonan"].items()})
        print("  allnan:", r["allnan"])
        if r["ptx"]: print("  ptx:", r["ptx"])

HEAD = subprocess.run("cd /content/triton && git rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip()
print("main HEAD", HEAD)
run_stage("main_baseline")
summarize("main_baseline")


In [ ]:
import subprocess
def pytest_nan(label):
    cmd = ("cd /content/triton/python/test/unit && python -m pytest language/test_core.py "
           "-k 'argmax_argmin_with_nan or tie_break_fast_with_nan' --import-mode=importlib -q -p no:cacheprovider --tb=short 2>&1 | tail -15")
    print(f"--- pytest NaN tests ({label})")
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1500).stdout)
pytest_nan("main_baseline")


In [ ]:
import re, subprocess
P = "/content/triton/python/triton/language/standard.py"
s = open(P).read()
if "value2 != value2" in s:
    print("already patched")
else:
    old_max = "    gt = value1 > value2 or tie\n"
    old_min = "    lt = value1 < value2 or tie\n"
    assert s.count(old_max) == 1 and s.count(old_min) == 1, ("unexpected source", s.count(old_max), s.count(old_min))
    new_max = old_max + (
        "    if core.constexpr(value1.dtype.is_floating()):\n"
        "        # never let a NaN win against a non-NaN value (tl.max/tl.min ignore NaN)\n"
        "        gt = gt or (value2 != value2 and value1 == value1)\n")
    new_min = old_min + (
        "    if core.constexpr(value1.dtype.is_floating()):\n"
        "        lt = lt or (value2 != value2 and value1 == value1)\n")
    s = s.replace(old_max, new_max).replace(old_min, new_min)
    open(P, "w").write(s)
print(subprocess.run("cd /content/triton && git diff -- python/triton/language/standard.py", shell=True, capture_output=True, text=True).stdout)
subprocess.run("rm -rf ~/.triton/cache", shell=True)


In [ ]:
run_stage("patched")
summarize("patched")
pytest_nan("patched")


In [ ]:
import subprocess
cmd = ("cd /content/triton/python/test/unit && python -m pytest language/test_core.py "
       "-k 'reduce or argm' --import-mode=importlib -q -p no:cacheprovider --tb=short --maxfail=10 2>&1 | tail -25")
print("--- pytest reduce/argm subset (patched)")
print(subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=3300).stdout)


In [ ]:
import json
print("main HEAD", HEAD)
print(json.dumps(RESULTS))


## 마지막 단계 (사람)
`파일 → GitHub에 사본 저장`에서 저장소 `alexxony/triton`, 브랜치 `colab-11993-verify`,
경로는 기본값(같은 파일)으로 저장한다. 이렇게 해야 실행 결과(출력)가 브랜치에 커밋돼 에이전트가 읽는다.
저장 후 `런타임 → 세션 종료`. 종료하지 않으면 컴퓨트 유닛이 계속 소모된다.
